# Set up PCA and GMM for clustering core Argo profiles

In [1]:
# os tools
import sys
import os
import os.path
import requests
import time
import urllib3
import shutil
from tqdm import tqdm

# data tools
import numpy                 as np
import pandas                as pd
import xarray                as xr
from   datetime              import date, datetime, timedelta                 # for saving figures with today's date
import datetime
import scipy
from   scipy.stats           import kruskal              # for boxenplot stats
from   scipy.stats           import mannwhitneyu         # for split violin plot stats
import gsw


# for all plots
import matplotlib
import matplotlib.pyplot     as plt                      # needed to make map setup
import matplotlib.colors     as colors
from   matplotlib.ticker     import EngFormatter         # for degree symbol in axis
from   cmocean               import cm as cmo
import seaborn               as     sns

# for map
import shapefile
import cartopy                                           # to make map
import matplotlib.path       as     mpath                # to draw circle for map
import cartopy.crs           as     ccrs                 # for map projection
import cartopy.feature       as     cfeature             # to add land features to map
# from   xhistogram.xarray     import histogram            # for map histogram
# from   mycolorpy             import colorlist as mcp     # to get n colors list
import pyproj  
import geopandas             as     gpd                  # for adding shapefiles of frontal zones 
from   osgeo                 import gdal
# import scikit_posthocs       as     sp                   # for stats

In [2]:
# Custom modules
import mod_cremas as crx 
import mod_ocean as myocn


from importlib import reload
import mod_plotting as myplt
# from mod_plotting import setup_SO_axes

plt.rcParams.update(myplt.my_params(size=12))

In [3]:
import shapefile
so_fronts = shapefile.Reader('./shapefiles/fronts/so_fronts.shp') 
stf_mod   = shapefile.Reader('./shapefiles/fronts/stf_mod/stf_mod.shp')

stf  = stf_mod.shape(0).points
saf  = so_fronts.shape(1).points
pf   = so_fronts.shape(2).points
sacc = so_fronts.shape(3).points
sie  = so_fronts.shape(4).points

max_latitude:          float = -30
add_gridlines:         bool  = True
color_land:            bool  = False
land_edgecolor:        str   = 'grey'
land_facecolor:        str   = 'grey'
fontsize:              float = 10
map_facecolor:         str   = 'white'
coast_linewidth:       float = 0.3
gridlines_linewidth:   float = 0.5
girdlines_color:       str   = 'grey'
gridlines_alpha:       float = 0.5
longitude_label_color: str   = 'grey'
latitude_label_color:  str   = 'grey'

In [4]:
# As of Apr 21 2025
filepath = '/Volumes/cremas-repo/data/bgc/L3-interp/'
bgcDS = xr.open_dataset(filepath + 'bgcDATA_valid_interp_2014-2023_acc20250313.nc')
bgcINDEX = xr.open_dataset(filepath + 'bgcINDEX_valid_interp_2014-2023_acc20250313.nc')

filepath = '/Volumes/cremas-repo/data/core/L3-interp/'
coreDS = xr.open_dataset(filepath + 'coreDATA_valid_interp_2014-2023_acc20250424.nc')
coreINDEX = xr.open_dataset(filepath + 'coreINDEX_valid_interp_2014-2023_acc20250424.nc')

filepath = '/Volumes/cremas-repo/data/socat/L2-mask/' 
# socat = pd.read_csv(filepath + 'SOCATv2024_SO_resampled_3h_median_acc20250121.csv')
socat = xr.open_dataset(filepath + 'SOCATv2024_SO_3h_open_ocean_INDEX_acc20250314.nc')
socat['yearday'] = myocn.datetime2ytd(socat['datetime'].astype('datetime64[ns]'), ref_time='2014-01-01')
socat = socat.where(socat.latitude<-35, drop=True)



In [5]:
coreDS

<xarray.Dataset>
Dimensions:      (profid: 328936, pressure: 197)
Coordinates:
  * profid       (profid) object '1900410_id260' ... '7901011_id005'
  * pressure     (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday      (profid, pressure) float64 ...
    latitude     (profid, pressure) float64 ...
    longitude    (profid, pressure) float64 ...
    wmoid        (profid, pressure) float64 ...
    datetime     (profid, pressure) datetime64[ns] ...
Data variables:
    CT           (profid, pressure) float64 ...
    SA           (profid, pressure) float64 ...
    sigma0       (profid, pressure) float64 ...
    spice        (profid, pressure) float64 ...
    temperature  (profid, pressure) float64 ...
    salinity     (profid, pressure) float64 ...
Attributes:
    title:            Core float profiles with valid surface data (QC flags 1...
    pressure_levels:  Pressure levels in dbar: [0, 5, 10, 15, 20, 25, 30, 35,...
    source:           Argopy, expert mode; GDAC snapshot Jan 2025
    date:             20250424

# 1.0 Scaling

In [6]:
from sklearn import preprocessing
import mod_pca as mypca

In [7]:
# Center but do not scale pressures separately, run time ~ 1min 
reload(mypca)
centered_coreDS = mypca.center_byPressure(coreDS, with_scaling = False)
centered_coreDS

# Scale and center, run time ~ 1min 
# reload(mypca)
# scaled_coreDS = mypca.center_byPressure(coreDS, with_scaling = True)
# scaled_coreDS


/opt/homebrew/Caskroom/mambaforge/base/envs/cremas/lib/python3.9/site-packages/xarray/coding/times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(
/opt/homebrew/Caskroom/mambaforge/base/envs/cremas/lib/python3.9/site-packages/xarray/coding/times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(
/opt/homebrew/Caskroom/mambaforge/base/envs/cremas/lib/python3.9/site-packages/xarray/coding/times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(
/opt/homebrew/Caskroom/mambaforge/base/envs/cremas/lib/python3.9/site-packages/xarray/coding/times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(
/opt/homebrew/Caskroom/mambaforge/base/envs/cremas/lib/p

<xarray.Dataset>
Dimensions:    (profid: 328936, pressure: 197)
Coordinates:
  * profid     (profid) object '1900410_id260' ... '7901011_id005'
  * pressure   (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday    (pressure, profid) float64 1.107 11.27 ... 3.633e+03 3.643e+03
    latitude   (pressure, profid) float64 -40.36 -40.11 -40.25 ... -52.84 -51.79
    longitude  (pressure, profid) float64 95.36 96.08 96.68 ... -48.95 -46.82
    wmoid      (pressure, profid) float64 1.9e+06 1.9e+06 ... 7.901e+06
    datetime   (pressure, profid) datetime64[ns] 2014-01-02T02:34:12 ... 2023...
Data variables:
    CT         (pressure, profid) float64 3.979 4.408 5.624 ... -1.205 -1.293
    SA         (pressure, profid) float64 0.4668 0.4658 0.4076 ... 0.1254 0.162
Attributes:
    title:            Core float profiles with valid surface data (QC flags 1...
    pressure_levels:  Pressure levels in dbar: [0, 5, 10, 15, 20, 25, 30, 35,...
    source:           Argopy, expert mode; GDAC snapshot Jan 2025
    date:             20250424

# 2.0 Use PCA to transform input core data

In [9]:
from   sklearn.decomposition import PCA
from   sklearn.decomposition import KernelPCA
from   sklearn               import manifold

In [10]:
# Stack temperature and salinity across pressure levels
temp_df = centered_coreDS.CT.transpose('profid', 'pressure').to_pandas()
sal_df  = centered_coreDS.SA.transpose('profid', 'pressure').to_pandas()

# Rename columns to distinguish them
plevels = centered_coreDS.pressure.values
temp_df.columns = [f'CT_{i}' for i in plevels]
sal_df.columns  = [f'SA_{i}' for i in plevels]

# Concatenate along columns
pca_input = (pd.concat([temp_df, sal_df], axis=1)
                # .drop(columns=['CT_0', 'SA_0']) 
                .dropna(axis=0) 
                ) 

In [ ]:
def run_pca(Xtrain, n_comp=2):
    """
    Fit and apply PCA to input data
    @param:  Xtrain:     Dataframe of Argo data to transform (indexed by profid)
                            (M, N) where M = # profiles, N = # pressure levels x 2 
             n_comp:         number of PCA components
    @return: Xtrans:        Dataframe of PCA transformed data
             mypca:          PCA object 
             comps:          dictionary of PCA components (eigenvectors length N)
    """
    print('===> Training PCA:')
    # Fit PCA 
    mypca = PCA(n_comp).fit(Xtrain) # PCA object
    labels = ['PC'+str(x+1) for x in range(n_comp)]
    comps = {k:v for k, v in zip(labels, mypca.components_)} # PCA components (eigenvectors)

    variance_total = np.sum(mypca.explained_variance_ratio_)
    print('# PCA components:', str(n_comp), '\nvariance explained:', str(variance_total))

    # Apply transformation to input data
    Xtrans = pd.DataFrame(mypca.transform(Xtrain), columns=comps.keys()) # Xtrans

    return Xtrans, mypca, comps


In [ ]:
# 1. === Set up parameters for single n_pca===
n_pca = 6
Xtrain = pca_input

# ==== 

[Xtrans, mypca, PCdict]= run_pca(Xtrain, n_comp=n_pca)



===> Training PCA:
# PCA components: 6 
variance explained: 0.9971482799109652


In [27]:
# # development === Set up parameters for single n_pca===
# n_pca = 6
# Xtrain = pca_input

# # train_frac=0.8
# # 
# # === Set up training data
# # nprofs = np.floor(train_frac * pca_input.shape[0]).astype(int)
# # use_rows = np.random.choice(pca_input.index.values, nprofs)
# # Xtrain = pca_input.loc[use_rows]

# # === Fit and apply ===
# core_pca = PCA(n_pca)
# core_pca.fit(Xtrain)

# # Transform entire input dataset into PCA representation
# Xpca = core_pca.transform(Xtrain)

# # Xtrans = pd.DataFrame(Xpca, columns=PCdict.keys())

# # Show variance
# variance_total = np.sum(core_pca.explained_variance_ratio_)
# print('# PCA components:', str(n_pca), '\nvariance explained:', str(variance_total))

# # Save component eigenvectors in a dictionary
# labs = ['PC'+str(x+1) for x in range(n_pca)]
# PCdict = {k:v for k, v in zip(labs, core_pca.components_)}

# # pca_components = pd.DataFrame(Xpca, columns=PCdict.keys()) # same as PCdict, but in Dataframe
# # PCdict

In [24]:
# 2. Run through different n_pca values
npc_Xtrans = {k:None for k in range(1, 11)}
npc_obj = {k:None for k in range(1, 11)}
npc_PCdict = {k:None for k in range(1, 11)}

for n_pca in range(1,11):
    [Xtrans, mypca, PCdict]= run_pca(Xtrain, n_comp=n_pca)

    npc_Xtrans[n_pca] = Xtrans
    npc_obj[n_pca] = mypca
    npc_PCdict[n_pca] = PCdict
    
# Xtrans = pd.DataFrame(Xpca, columns=PCdict.keys()) # transformed data
# Xtrans

===> Training PCA:
# PCA components: 1 
variance explained: 0.9352295270746469
===> Training PCA:
# PCA components: 2 
variance explained: 0.9759597706289914
===> Training PCA:
# PCA components: 3 
variance explained: 0.9900160968223088
===> Training PCA:
# PCA components: 4 
variance explained: 0.9939163740655287
===> Training PCA:
# PCA components: 5 
variance explained: 0.9958325159831443
===> Training PCA:
# PCA components: 6 
variance explained: 0.9971482799109652
===> Training PCA:
# PCA components: 7 
variance explained: 0.9979490864771017
===> Training PCA:
# PCA components: 8 
variance explained: 0.9983829569762356
===> Training PCA:
# PCA components: 9 
variance explained: 0.998699584312414
===> Training PCA:
# PCA components: 10 
variance explained: 0.9989548433915106


- `Xtrain` : Untransformed input dataset (float profiles are rows)
- `Xpca` : Transformed input dataset in PCA representation
- `core_pca`: pca object
- `PCdict`: dictionary of PC components (keys are PC1, PC2...)

In [26]:
# Profiles now represented as linear combinations of PCA components
Xtrans

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,52.003264,-13.911040,1.827904,2.437100,-0.624407,1.974620,-1.010184,2.553272,0.153580,-0.863795
1,55.507774,-15.915819,2.691364,4.098044,-0.503678,0.526700,-1.870078,1.100428,0.020072,-0.351368
2,56.581794,-16.191437,3.831421,2.452300,0.172906,1.200269,-0.307096,2.025573,0.135869,-0.123830
3,58.423547,-14.316440,4.377420,2.258251,0.350651,2.197945,-0.299931,1.816412,-0.175800,-0.813231
4,58.468549,-14.497622,5.589924,1.105045,0.654300,2.345868,1.540735,1.161874,-0.016192,-0.727187
...,...,...,...,...,...,...,...,...,...,...
313616,-53.705715,0.162641,4.595398,-0.276967,1.042229,-1.187716,-2.732722,-1.424386,0.883177,0.075850
313617,-47.182295,5.026448,4.699122,-2.117196,1.218738,-0.393413,0.767217,-0.265938,1.097323,0.305916
313618,-47.567601,1.521335,2.334249,-0.898852,0.591323,1.569011,-0.042219,-0.509462,0.161477,0.422690
313619,-47.993736,3.588269,4.559838,-2.241074,0.826065,1.341585,0.754445,-0.664344,0.402124,-1.005225
